**Requerimientos**


In [0]:
%pip install pyyaml

**Cargando config.yml**

In [0]:
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import lit, col, to_timestamp
import yaml

with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)
print("✅ Configuración cargada")

**Creando esquema silver**

**Leyendo tabla bronze**

In [0]:
silver_df = spark.table(config["tables"]["silver_table"])
display(silver_df.limit(5))
print("✅ Datatable cargada")

In [0]:
# ¿Cuántos registros tiene silver en total?
print("Total silver:", silver_df.count())

# ¿Cuántos por ciudad?
silver_df.groupBy("Location").count().orderBy("count", ascending=False).show()

# ¿Hay duplicados?
from pyspark.sql.functions import count, col

total     = silver_df.count()
distintos = silver_df.dropDuplicates().count()
print(f"Total: {total} | Distintos: {distintos} | Duplicados: {total - distintos}")

In [0]:
from pyspark.sql.functions import col, month, year, round, avg, max, min, count

# ── 1. Promedio de temperatura por ciudad y mes ──────────────────────────────
avg_temp_df = silver_df \
    .withColumn("year", year(col("Date_Time"))) \
    .withColumn("month", month(col("Date_Time"))) \
    .groupBy("Location", "year", "month") \
    .agg(
        round(avg("Temperature_C"), 2).alias("avg_temp_C"),
        round(avg("Humidity_pct"), 2).alias("avg_humidity_pct"),
        round(avg("Precipitation_mm"), 2).alias("avg_precipitation_mm"),
        round(avg("Wind_Speed_kmh"), 2).alias("avg_wind_kmh")
    ) \
    .orderBy("Location", "year", "month")

print("=== Promedio por ciudad y mes ===")
#display(avg_temp_df)


# ── 2. Días con clima extremo ────────────────────────────────────────────────
extreme_df = silver_df.filter(
    (col("Temperature_C") > 35) |          # calor extremo
    (col("Temperature_C") < -10) |         # frío extremo
    (col("Humidity_pct") > 90) |           # humedad muy alta
    (col("Precipitation_mm") > 8) |        # lluvia intensa
    (col("Wind_Speed_kmh") > 25)           # viento fuerte
) \
.withColumn("extreme_heat",    col("Temperature_C") > 35) \
.withColumn("extreme_cold",    col("Temperature_C") < -10) \
.withColumn("high_humidity",   col("Humidity_pct") > 90) \
.withColumn("heavy_rain",      col("Precipitation_mm") > 8) \
.withColumn("strong_wind",     col("Wind_Speed_kmh") > 25) \
.orderBy("Location", "Date_Time")

print("=== Días con clima extremo ===")
#display(extreme_df)


# ── 3. Resumen de extremos por ciudad ────────────────────────────────────────
extreme_summary_df = extreme_df \
    .groupBy("Location") \
    .agg(
        count("*").alias("extreme_days"),
        round(max("Temperature_C"), 2).alias("max_temp_C"),
        round(min("Temperature_C"), 2).alias("min_temp_C"),
        round(max("Precipitation_mm"), 2).alias("max_precipitation_mm"),
        round(max("Wind_Speed_kmh"), 2).alias("max_wind_kmh")
    ) \
    .orderBy(col("extreme_days").desc())

print("=== Resumen de extremos por ciudad ===")
#display(extreme_summary_df)

In [0]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── Datos desde silver_df ────────────────────────────────────────────────────
from pyspark.sql.functions import col, month, avg, round as spark_round

# Promedio por ciudad
avg_df = silver_df.groupBy("Location").agg(
    spark_round(avg("Temperature_C"), 2).alias("avg_temp"),
    spark_round(avg("Humidity_pct"), 2).alias("avg_humidity"),
    spark_round(avg("Precipitation_mm"), 2).alias("avg_precip"),
    spark_round(avg("Wind_Speed_kmh"), 2).alias("avg_wind")
).orderBy("avg_temp").toPandas()

# Todos los registros para scatter
scatter_df = silver_df.select("Location", "Temperature_C", "Humidity_pct").toPandas()

# ── Colores por temperatura ──────────────────────────────────────────────────
def temp_color(t):
    if t > 30:   return "#D85A30"
    elif t < -5: return "#378ADD"
    else:        return "#1D9E75"

bar_colors   = [temp_color(t) for t in avg_df["avg_temp"]]
scatter_colors = [temp_color(t) for t in scatter_df["Temperature_C"]]

# ── Figura ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 10), facecolor="white")
fig.suptitle("Weather Silver Layer — Dashboard", fontsize=14, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# 1. Temperatura promedio por ciudad
ax1 = fig.add_subplot(gs[0, :])
bars = ax1.bar(avg_df["Location"], avg_df["avg_temp"], color=bar_colors, edgecolor="none", width=0.6)
ax1.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax1.set_title("temperatura promedio por ciudad (°C)", fontsize=11)
ax1.set_ylabel("°C", fontsize=10)
ax1.tick_params(axis="x", rotation=20, labelsize=9)
ax1.tick_params(axis="y", labelsize=9)
ax1.spines[["top", "right"]].set_visible(False)
for bar, val in zip(bars, avg_df["avg_temp"]):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + (0.5 if val >= 0 else -1.5),
             f"{val:.1f}°", ha="center", va="bottom", fontsize=8)

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#D85A30", label="calor extremo (>30°C)"),
    Patch(facecolor="#1D9E75", label="normal"),
    Patch(facecolor="#378ADD", label="frío (<-5°C)")
]
ax1.legend(handles=legend_elements, fontsize=8, loc="upper left")

# 2. Scatter temperatura vs humedad
ax2 = fig.add_subplot(gs[1, 0])
ax2.scatter(scatter_df["Temperature_C"], scatter_df["Humidity_pct"],
            c=scatter_colors, alpha=0.8, s=60, edgecolors="none")
ax2.set_title("temperatura vs humedad", fontsize=11)
ax2.set_xlabel("temperatura (°C)", fontsize=9)
ax2.set_ylabel("humedad (%)", fontsize=9)
ax2.tick_params(labelsize=9)
ax2.spines[["top", "right"]].set_visible(False)

# 3. Precipitación
ax3 = fig.add_subplot(gs[1, 1])
ax3.barh(avg_df["Location"], avg_df["avg_precip"], color="#378ADD", edgecolor="none")
ax3.set_title("precipitación promedio (mm)", fontsize=11)
ax3.set_xlabel("mm", fontsize=9)
ax3.tick_params(labelsize=9)
ax3.spines[["top", "right"]].set_visible(False)

plt.savefig("/tmp/weather_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Guardado en /tmp/weather_dashboard.png")